# Human-in-the-Loop

This notebook introduces **Human-in-the-Loop (HITL)** — a safety mechanism that **pauses** the agent before it executes a specific tool call, waits for a human to approve or reject it, and then either continues or cancels.

## Key concepts

- **`HumanInTheLoopMiddleware`** – A middleware that intercepts tool calls specified in `interrupt_on` and suspends agent execution, waiting for human input.
- **`interrupt_on`** – A dict mapping tool names to `True`/`False` flags that control which tools trigger a pause.
- **`__interrupt__`** – The key in the agent's result dict that contains the pending approval requests when the agent is paused.
- **`Command(resume=...)`** – The object you pass to `.invoke()` to resume a paused agent. You supply decisions (approve/reject) for each pending interrupt.
- **`thread_id`** – Required when using HITL. The same `thread_id` must be used for both the initial call and the resume call — they are part of the same conversation thread.

## The Human-in-the-Loop flow

```
1. User sends message  →  Agent decides to call a tool
2. HumanInTheLoopMiddleware intercepts  →  Agent PAUSES, returns __interrupt__
3. Human reviews the pending tool call  →  Approves or rejects
4. Human sends Command(resume=...) with decision  →  Agent CONTINUES
5. If approved: tool runs normally
   If rejected: agent receives a "User rejected..." message instead of a tool result
```

## Why is this important?

Some tool calls are **irreversible** (sending emails, booking flights, deleting data). HITL lets you review and authorize such actions before they happen.

In [ ]:
# Install required packages.
!pip install -q langchain langchain-openai

In [ ]:
from IPython.display import HTML           # For rendering styled HTML in Jupyter output cells
from google.colab import userdata
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware  # The HITL middleware
from langchain.messages import HumanMessage
from langchain.tools import tool
from langchain_core.messages import BaseMessage
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver  # Required for HITL (state must be saved between pause and resume)
from langgraph.types import Command, Interrupt          # Command: used to resume; Interrupt: the pause object
from pydantic import SecretStr
from typing import List

openai_api_key = SecretStr(userdata.get('OPENAI_API_KEY'))

def print_conversation(conversation: List[BaseMessage]):
    for message in conversation:
        message.pretty_print()

# Helper to display pending interrupt requests in a styled HTML box.
# Each interrupt contains an "action_requests" list describing what the agent wants to do.
def print_interrupts(interrupts: List[Interrupt]):
    for interrupt in interrupts:
        for action_request in interrupt.value["action_requests"]:
            # Renders a red-bordered box showing the pending tool call description.
            display(HTML(f'<div style="border: 1px dashed red; margin: 5px; padding: 10px; white-space: pre-wrap;">{action_request["description"]}</div>'))

In [ ]:
# Define a "dangerous" tool that sends a real email.
# This is the kind of tool where you'd want human approval before execution,
# because sending emails is an irreversible side effect.
@tool
def send_email(recipient_email: str, customer_name: str, offer: str) -> str:
    """Call this tool to send a promotional email to a customer."""
    return f"Sent promo email to {customer_name} <{recipient_email}> about: {offer}"

In [ ]:
# Build the agent with HumanInTheLoopMiddleware.
# Note: a checkpointer (InMemorySaver) is REQUIRED for HITL to work.
# The agent must save its state before pausing so it can resume from the same point.
agent = create_agent(
    model=ChatOpenAI(model="gpt-5-nano", api_key=openai_api_key, reasoning_effort="low"),
    tools=[send_email],
    system_prompt=(
        "You are a careful marketing assistant. "
        "Use tools only when the user clearly asks for a real-world action."
    ),
    checkpointer=InMemorySaver(),  # REQUIRED: saves state so the agent can be resumed after the interrupt
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                send_email.name: True  # Pause before EVERY call to send_email
            }
        )
    ],
)

In [ ]:
# Send the initial user message.
# The agent will:
#   1. Decide to call send_email with the right parameters.
#   2. HumanInTheLoopMiddleware intercepts — PAUSES execution.
#   3. Return a result dict that includes an "__interrupt__" key with the pending approval request.
#
# NOTE: The result WILL NOT contain the email being sent yet — it's waiting for your approval.
result = agent.invoke(
    input={
        "messages": [HumanMessage("Send a friendly promotional email to Maria at maria@example.com about a weekend-only 20% sneaker sale. Mention that the code is SPRING20.")]
    },
    config={
        "configurable": {
            "thread_id": "promotional_email_1"  # Thread ID — needed to resume this exact run later
        }
    }
)

In [ ]:
# Print the conversation so far (no tool result yet — agent is paused).
print_conversation(result["messages"])

# Print the pending interrupt requests in a styled HTML box.
# This shows you WHAT the agent wants to do — review it before deciding to approve or reject.
print_interrupts(result.get("__interrupt__", []))

In [ ]:
# Resume the paused agent and APPROVE the pending tool call.
# IMPORTANT: We must use the SAME thread_id as the original call ("promotional_email_1").
# This tells LangGraph to load the saved state and continue from where it paused.
#
# The `decisions` list corresponds to the list of pending interrupts.
# Each decision is an object with a "type" field:
#   - { "type": "approve" }  → allow the tool call to proceed
#   - { "type": "reject" }   → cancel the tool call; the agent receives a rejection message instead
continuation = agent.invoke(
    input=Command(resume={ "decisions": [{ "type": "approve" }] }),  # Approve the email send
    config={
        "configurable": {
            "thread_id": "promotional_email_1"  # Must match the original thread to resume it
        }
    }
)

In [ ]:
# Print the final conversation after approval and tool execution.
# You should now see the tool result (email sent confirmation) and the agent's final response.
print_conversation(continuation["messages"])